# 00 - Prepare the data

Builds the training dataset from the two labeled sources and then splits it **once**, so every model is trained, tuned and tested on exactly the same rows.

| Source | What it is |
|---|---|
| `data/raw/semeval_train.csv` | SemEval laptop reviews with aspect terms and polarity |
| `data/raw/reddit_labeled.csv` | Reddit comments from our own extension, aspects and polarity labeled by hand |

**Output:**
- `data/processed/combined_train.csv` with columns `text`, `aspect_term`, `polarity`, `source`, `group_id`
- `data/splits/train_split.csv`, `validation_split.csv`, `test_split.csv` (about 80% / 10% / 10%)
- `data/splits/split_report.json` with the counts and a short fingerprint of the data

The text is kept as natural sentences (original case, stopwords included), because that is what the app sends to the models. Long Reddit comments are cut to a window around the aspect, because the models only read the first ~100 words.

**The split keeps whole sentences together.** One sentence appears in several rows, one per aspect. A plain random split could put "the battery is great but the cord is awful" in training and the same sentence with a different aspect in the test set. The model would have seen the test sentence. Rows from the same sentence (or the same original Reddit comment) always land in the same part.

Re-running this notebook gives the same split as long as the data and the seed are unchanged. If you change the data, retrain every model afterwards.

In [9]:
import hashlib
import json
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from opinion_owl.aspect_patterns import COMPILED
from opinion_owl.config import (
    COMBINED_TRAIN, REDDIT_LABELED, SEMEVAL_TRAIN,
    SPLIT_REPORT, TEST_SPLIT, TRAIN_SPLIT, VALIDATION_SPLIT,
)

SOURCES = {"semeval": SEMEVAL_TRAIN, "reddit": REDDIT_LABELED}

LABEL_FIXES = {"postive": "positive", "po": "positive"}  # typos found in the raw labels
VALID_LABELS = {"negative", "neutral", "positive", "conflict"}

MAX_WORDS = 80           # longer rows are cut to a window of this many words around the aspect
DROP_UNFOUND = True      # drop long rows where the aspect cannot be located in the text
DROP_CONFLICTING = True  # drop (sentence, aspect) pairs that carry two different labels

SEED = 42
N_FOLDS = 10             # 10 folds: one for test, one for validation, eight for training

## 1. Load both sources

In [10]:
frames = []
for source, path in SOURCES.items():
    frame = pd.read_csv(path)
    missing_columns = {"text", "aspect_term", "polarity"} - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {', '.join(sorted(missing_columns))}")

    frame = frame[["text", "aspect_term", "polarity"]].copy()
    frame["source"] = source
    frames.append(frame)
    print(f"{source}: {len(frame)} rows")

data = pd.concat(frames, ignore_index=True)
print("Total before cleaning:", len(data))

semeval: 2358 rows
reddit: 376 rows
Total before cleaning: 2734


## 2. Clean

In [11]:
start = len(data)

# Reddit comments contain line breaks; squash all whitespace so each row is one line.
data["text"] = data["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
data["aspect_term"] = data["aspect_term"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip().str.lower()
data["polarity"] = data["polarity"].astype(str).str.strip().str.lower().replace(LABEL_FIXES)

# Rows whose text or aspect were empty cells became the string "nan" after astype(str).
empty = data["text"].isin(["", "nan"]) | data["aspect_term"].isin(["", "nan"])
bad_label = ~data["polarity"].isin(VALID_LABELS)
print(f"Dropped {int(empty.sum())} rows with empty text/aspect and {int(bad_label.sum())} with an unknown label")
data = data[~empty & ~bad_label].reset_index(drop=True)

# Remember which original sentence or comment every row came from. This must be set BEFORE
# long comments are cut into windows, because after that two rows from one comment have
# different text. The split in step 6 uses it to keep such rows together.
data["group_id"] = pd.factorize(data["source"] + "|" + data["text"].str.lower())[0]

Dropped 0 rows with empty text/aspect and 2 with an unknown label


## 3. Shorten long comments

In [12]:
# The models read only the first ~100 words / 128 tokens. In a long Reddit comment the
# aspect can sit after that cut, and then the label no longer matches what the model sees.
# So rows longer than MAX_WORDS are cut to a window around the place the aspect is mentioned.


def find_mention(text, aspect):
    """Return (character position, how it was found) for the aspect in the text.

    Reddit aspect terms are often "product + category" (for example "hp omnibook 3 ram"),
    so the end of the term is searched first, since that is the part people write about.
    """
    words = aspect.split()

    match = re.search(re.escape(aspect), text, flags=re.IGNORECASE)
    if match:
        return match.start(), "exact"

    # 1. the longest end part that is one of our aspect categories ("ram", "build quality", ...)
    for n in range(len(words)):
        for pattern in COMPILED.get(" ".join(words[n:]), []):
            match = pattern.search(text)
            if match:
                return match.start(), "category"

    # 2. the end part as plain words ("storage" from "pc storage")
    for n in range(1, len(words)):
        match = re.search(re.escape(" ".join(words[n:])), text, flags=re.IGNORECASE)
        if match:
            return match.start(), "tail"

    # 3. the start of a long word, starting from the end ("repairability" -> "repai")
    for word in reversed(words):
        if len(word) >= 4:
            match = re.search(r"\b" + re.escape(word[:5]), text, flags=re.IGNORECASE)
            if match:
                return match.start(), "word"

    return None, "not_found"


def focus_on_aspect(text, aspect, max_words=MAX_WORDS):
    """Keep the part of a long comment that talks about the aspect."""
    words = text.split()
    if len(words) <= max_words:
        return text, "kept"

    start_char, how = find_mention(text, aspect)
    if start_char is None:
        return text, "not_found"

    position = len(text[:start_char].split())
    start = max(0, min(position - max_words // 2, len(words) - max_words))
    return " ".join(words[start:start + max_words]), f"windowed_{how}"


results = [focus_on_aspect(t, a) for t, a in zip(data["text"], data["aspect_term"])]
data["text"] = [r[0] for r in results]
status = pd.Series([r[1] for r in results], index=data.index)
print(status.value_counts())

if DROP_UNFOUND:
    data = data[status != "not_found"]
    print("Dropped long rows without a findable aspect.")
data = data.reset_index(drop=True)

kept                 2666
windowed_category      33
windowed_tail           9
windowed_exact          9
windowed_word           8
not_found               7
Name: count, dtype: int64
Dropped long rows without a findable aspect.


## 4. Remove duplicates and contradictions

In [13]:
# The same (sentence, aspect, label) listed twice adds no information.
before = len(data)
data = data.drop_duplicates(["text", "aspect_term", "polarity"]).reset_index(drop=True)
print(f"Dropped {before - len(data)} exact duplicate rows")

# The same (sentence, aspect) with two different labels is a labeling disagreement.
# Such a pair is ambiguous for training, and unfair for testing if its two copies land in different splits.
conflicting = data.groupby(["text", "aspect_term"])["polarity"].transform("nunique") > 1
print("Rows in conflicting pairs:", int(conflicting.sum()))
if DROP_CONFLICTING:
    data = data[~conflicting].reset_index(drop=True)
    print("Dropped them.")

print(f"Rows kept: {len(data)} of {start}")

Dropped 53 exact duplicate rows
Rows in conflicting pairs: 30
Dropped them.
Rows kept: 2642 of 2734


## 5. Check and save the dataset

In [14]:
print(pd.crosstab(data["source"], data["polarity"], margins=True))

words = data["text"].str.split().str.len()
print("\nWords per row:")
print(words.groupby(data["source"]).describe(percentiles=[0.5, 0.95])[["count", "50%", "95%", "max"]])
print("Rows over", MAX_WORDS, "words:", int((words > MAX_WORDS).sum()), "(should be 0 unless DROP_UNFOUND is False)")

# index=False: no extra index columns (the old file had two "Unnamed" columns).
data.to_csv(COMBINED_TRAIN, index=False)
print("\nSaved", len(data), "rows to", COMBINED_TRAIN)

polarity  conflict  negative  neutral  positive   All
source                                               
reddit           0       151       13       202   366
semeval         43       826      444       963  2276
All             43       977      457      1165  2642

Words per row:
          count   50%   95%   max
source                           
reddit    366.0  35.5  80.0  80.0
semeval  2276.0  17.0  37.0  78.0
Rows over 80 words: 0 (should be 0 unless DROP_UNFOUND is False)

Saved 2642 rows to C:\Programming\opinion-owl\data\processed\combined_train.csv


## 6. Split once: train / validation / test

In [15]:
# StratifiedGroupKFold does two things at once:
#  - group: all rows with the same group_id go to the same fold (no sentence on both sides)
#  - stratify: each fold gets about the same mix of source and polarity, so the small Reddit
#    part and the rare "conflict" class show up in every split
strata = data["source"] + "_" + data["polarity"]
folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_of_row = np.full(len(data), -1)
for fold, (_, rows) in enumerate(folds.split(data, strata, groups=data["group_id"])):
    fold_of_row[rows] = fold

data["split"] = np.where(fold_of_row == 0, "test", np.where(fold_of_row == 1, "validation", "train"))

parts = {name: data[data["split"] == name].drop(columns="split").reset_index(drop=True)
         for name in ("train", "validation", "test")}

# Safety check: no sentence may appear in two parts.
group_sets = {name: set(part["group_id"]) for name, part in parts.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    shared = group_sets[a] & group_sets[b]
    assert not shared, f"{len(shared)} sentences appear in both {a} and {b}"
print("No sentence appears in more than one part.")

print("\nRows per part:", {name: len(part) for name, part in parts.items()})
print(pd.crosstab([data["source"], data["polarity"]], data["split"])[["train", "validation", "test"]])

No sentence appears in more than one part.

Rows per part: {'train': 2113, 'validation': 264, 'test': 265}
split             train  validation  test
source  polarity                         
reddit  negative    121          15    15
        neutral      10           1     2
        positive    161          20    21
semeval conflict     35           4     4
        negative    660          83    83
        neutral     355          45    44
        positive    771          96    96


In [16]:
parts["train"].to_csv(TRAIN_SPLIT, index=False)
parts["validation"].to_csv(VALIDATION_SPLIT, index=False)
parts["test"].to_csv(TEST_SPLIT, index=False)

# A short fingerprint of the dataset, so any set of results can be tied to the data behind it.
fingerprint = hashlib.md5(COMBINED_TRAIN.read_bytes()).hexdigest()[:10]

report = {
    "dataset_fingerprint": fingerprint,
    "seed": SEED,
    "folds": N_FOLDS,
    "max_words": MAX_WORDS,
    "rows": {name: len(part) for name, part in parts.items()},
    "rows_by_source": {
        name: part["source"].value_counts().to_dict() for name, part in parts.items()
    },
    "rows_by_polarity": {
        name: part["polarity"].value_counts().to_dict() for name, part in parts.items()
    },
}
with open(SPLIT_REPORT, "w") as f:
    json.dump(report, f, indent=2)

print("Saved the three split files and", SPLIT_REPORT.name)
print("Dataset fingerprint:", fingerprint)

Saved the three split files and split_report.json
Dataset fingerprint: 938439a781
